# Data Science Tool Mastery Project

Practical proficiency project covering Python, Jupyter, NumPy, Pandas, Matplotlib, Seaborn, SciPy, SQL/SQLite, and scikit-learn. Each tool is demonstrated through an exercise and then integrated into a portfolio customer-churn workflow.

The included customer dataset is **synthetic** and contains no real customer information.

## 1. Environment Verification

Install dependencies with `pip install -r requirements.txt` and launch with `jupyter notebook`.

In [ ]:
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, scipy, sklearn
print("Python:",sys.version.split()[0]); print("NumPy:",np.__version__); print("Pandas:",pd.__version__); print("SciPy:",scipy.__version__); print("scikit-learn:",sklearn.__version__); print("Seaborn:",sns.__version__)

## 2. NumPy Mastery

Reproducible random generation, vectorized transformations, statistics, percentiles, and boolean filtering.

In [ ]:
rng=np.random.default_rng(42); sales=np.clip(rng.normal(500,120,1000),50,None)
print("Mean:",sales.mean(),"Median:",np.median(sales),"Std:",sales.std(),"P75:",np.percentile(sales,75)); print("Above P75:",(sales>np.percentile(sales,75)).sum()); print("Log transform:",np.log1p(sales[:5]))

## 3. Pandas Mastery

Load, inspect, clean, engineer features, aggregate, and pivot a customer dataset.

In [ ]:
df=pd.read_csv("data/customer_churn_practice.csv"); print("Shape:",df.shape); display(df.head()); display(df.describe(include="all").T)
print("Missing:"); display(df.isna().sum().to_frame("missing")); print("Duplicates:",df.duplicated().sum())
clean_df=df.drop_duplicates().copy()
for c in ["satisfaction","monthly_spend"]: clean_df[c]=clean_df[c].fillna(clean_df[c].median())
clean_df["spend_per_tenure_month"]=clean_df["monthly_spend"]/clean_df["tenure_months"].clip(lower=1)
display(clean_df.groupby("plan").agg(customers=("customer_id","count"),avg_spend=("monthly_spend","mean"),avg_satisfaction=("satisfaction","mean"),churn_rate=("churn","mean")).sort_values("churn_rate",ascending=False))
display(pd.pivot_table(clean_df,values="monthly_spend",index="region",columns="plan",aggfunc="mean"))

## 4. Matplotlib Mastery

In [ ]:
plt.figure(figsize=(9,5)); plt.hist(clean_df.monthly_spend,bins=30); plt.title("Monthly Spend Distribution"); plt.xlabel("Monthly spend"); plt.ylabel("Customers"); plt.tight_layout(); plt.show()
x=clean_df.groupby("region").monthly_spend.mean().sort_values(ascending=False); plt.figure(figsize=(8,5)); plt.bar(x.index,x.values); plt.title("Average Spend by Region"); plt.xlabel("Region"); plt.ylabel("Average spend"); plt.tight_layout(); plt.show()

## 5. Seaborn Mastery

In [ ]:
plt.figure(figsize=(9,5)); sns.boxplot(data=clean_df,x="plan",y="monthly_spend"); plt.title("Monthly Spend by Plan"); plt.tight_layout(); plt.show()
plt.figure(figsize=(9,6)); sns.scatterplot(data=clean_df,x="satisfaction",y="monthly_spend",hue="churn",alpha=.7); plt.title("Satisfaction vs Spend"); plt.tight_layout(); plt.show()
plt.figure(figsize=(10,7)); sns.heatmap(clean_df[["age","monthly_spend","tenure_months","support_tickets","satisfaction","churn"]].corr(),annot=True,fmt=".2f"); plt.title("Correlation Heatmap"); plt.tight_layout(); plt.show()

## 6. SciPy Mastery — Hypothesis Testing

Question: is mean satisfaction different between churned and retained customers? Use Welch's independent two-sample t-test at α=0.05.

In [ ]:
from scipy import stats
a=clean_df.loc[clean_df.churn==1,"satisfaction"]; b=clean_df.loc[clean_df.churn==0,"satisfaction"]; t,p=stats.ttest_ind(a,b,equal_var=False)
print("Churned mean:",a.mean(),"Retained mean:",b.mean(),"t:",t,"p:",p); print("Decision:","Reject H0" if p<.05 else "Fail to reject H0")

## 7. SQL / SQLite Mastery

Load the cleaned table into SQLite and practice SELECT, COUNT, AVG, GROUP BY, ROUND, and ORDER BY.

In [ ]:
import sqlite3
conn=sqlite3.connect(":memory:"); clean_df.to_sql("customers",conn,index=False,if_exists="replace")
q="SELECT plan, COUNT(*) AS customers, ROUND(AVG(monthly_spend),2) AS avg_spend, ROUND(AVG(churn)*100,2) AS churn_rate_percent FROM customers GROUP BY plan ORDER BY churn_rate_percent DESC;"
display(pd.read_sql_query(q,conn))

## 8. scikit-learn Mastery — Portfolio Churn Classifier

Build a reproducible Logistic Regression pipeline with train/test split, imputation, scaling, one-hot encoding, and multiple evaluation metrics.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix,classification_report
features=["age","monthly_spend","tenure_months","support_tickets","satisfaction","plan","region"]; X=clean_df[features]; y=clean_df.churn
num=["age","monthly_spend","tenure_months","support_tickets","satisfaction"]; cat=["plan","region"]
prep=ColumnTransformer([("num",Pipeline([("imputer",SimpleImputer(strategy="median")),("scaler",StandardScaler())]),num),("cat",OneHotEncoder(handle_unknown="ignore"),cat)])
model=Pipeline([("preprocessor",prep),("classifier",LogisticRegression(max_iter=1000,random_state=42))])
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.2,stratify=y,random_state=42); model.fit(Xtr,ytr); pred=model.predict(Xte); proba=model.predict_proba(Xte)[:,1]
scores={"accuracy":accuracy_score(yte,pred),"precision":precision_score(yte,pred,zero_division=0),"recall":recall_score(yte,pred,zero_division=0),"f1":f1_score(yte,pred,zero_division=0),"roc_auc":roc_auc_score(yte,proba)}; display(pd.Series(scores,name="score").to_frame()); print(classification_report(yte,pred,zero_division=0))

In [ ]:
cm=confusion_matrix(yte,pred); plt.figure(figsize=(6,5)); sns.heatmap(cm,annot=True,fmt="d",cbar=False); plt.title("Churn Classifier Confusion Matrix"); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.tight_layout(); plt.show()

## 9. Tool Integration and Portfolio Interpretation

**Pandas → cleaning → feature engineering → Matplotlib/Seaborn → SciPy → SQL → scikit-learn → evaluation**

The classifier is a baseline educational model, not a production churn system. A production workflow would add cross-validation, tuning, threshold optimization, explainability, monitoring, privacy/governance, and drift detection.

### Completion Checklist
- [x] Python/Jupyter environment
- [x] NumPy
- [x] Pandas
- [x] Matplotlib
- [x] Seaborn
- [x] SciPy
- [x] SQL/SQLite
- [x] scikit-learn
- [x] Practical exercises
- [x] Portfolio mini-project
- [x] Reproducibility and responsible interpretation

## Conclusion

The project demonstrates tool mastery through practical use rather than passive library descriptions. It connects numerical computing, tabular analysis, visualization, statistics, SQL, and machine learning into a reproducible portfolio workflow.